# 05 · REVEL — a supervised meta-predictor & the *circularity* problem

**REVEL** (*Rare Exome Variant Ensemble Learner*, Ioannidis et al. 2016, *AJHG*, PMID 27666373) is a **supervised** random-forest **ensemble** of 13 predictors, trained on curated pathogenic/benign variants. That training is powerful — but it makes benchmarking REVEL against ClinVar **partly circular**, which is the key lesson here.

> ⚠️ **DEMO DATA.** The REVEL scores here are hand-authored illustrative values for ~13 curated CFTR variants — **not** real REVEL output. Every table keeps a `source` column reading `DEMO`. See the *How to get REAL data* box, then join real per-variant scores on `protein_variant`; the code runs unchanged once `source` is `REAL`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
%matplotlib inline

## 2 · REVEL — what is it?

**REVEL** = *Rare Exome Variant Ensemble Learner* (Ioannidis et al. 2016, *American Journal of
Human Genetics*, **PMID 27666373**).

Think of REVEL as a **committee vote of 13 other predictors**. Instead of inventing a brand-new way
to judge a missense variant, its authors took the scores of **13 individual tools** (e.g. SIFT,
PolyPhen-2, MutationTaster, and several conservation scores) and trained a **random forest** — a
supervised machine-learning model — to combine them into one number.

Key facts to remember:

| Property | REVEL |
|---|---|
| Learning type | **Supervised** (learned from labelled examples) |
| Model | Random-forest **ensemble** of 13 component predictors |
| Trained on | Curated **pathogenic** and **benign** missense variants |
| Score range | **0 → 1** (higher = more likely pathogenic) |
| Covers | Missense variants only |

The "supervised" part is crucial: someone had to **hand REVEL a training set of variants already
labelled pathogenic or benign**. Where did those labels come from? Databases like **ClinVar** and
**HGMD**. Hold that thought — it is the seed of the circularity problem in section 3.


In [2]:
# Load the DEMO REVEL table. Note the `source` column says DEMO on every row.
revel = tk.load_revel()
print("REVEL score range in this DEMO table:", revel.revel_score.min(), "→", revel.revel_score.max())
revel

REVEL score range in this DEMO table: 0.1 → 0.94


,protein_variant,revel_score,source
0,G551D,0.940,DEMO
2,R117H,0.520,DEMO
3,R334W,0.880,DEMO
4,G85E,0.900,DEMO
5,D1152H,0.672,DEMO
6,R668C,0.220,DEMO
7,Tyr161Cys,0.872,DEMO
8,Gly970Asp,0.812,DEMO
9,Ser912Leu,0.782,DEMO
10,Val520Phe,0.755,DEMO


## 3 · ⚠️ THE CIRCULARITY WARNING — the key idea in this notebook

> ### 🔁 Why "REVEL disagrees with ClinVar" can be *misleading*
>
> REVEL was **trained on labels that share lineage with ClinVar and HGMD.**
>
> So when you benchmark REVEL against ClinVar and find they **agree**, that agreement may be
> **partly baked in** — REVEL may simply be *remembering* variants it was trained on. This is
> **label leakage**: information from the "answer key" leaked into the model.
>
> And when REVEL **disagrees** with ClinVar, that disagreement is **not guaranteed to be
> independent evidence** either — it can reflect quirks of the training labels rather than a fresh,
> orthogonal opinion.
>
> **Bottom line:** a supervised predictor benchmarked against its own training-label source is
> **partly circular**. It looks more accurate than it truly is on *new* variants.

**Contrast with the unsupervised tools from notebooks 03-04:**

| Tool | Learned from clinical labels? | Benchmarking vs ClinVar is… |
|---|---|---|
| AlphaMissense, EVE, ESM1b | **No** (sequence / structure / evolution only) | **Fair-ish** — an independent opinion |
| **REVEL** | **Yes** (ClinVar/HGMD lineage) | **Partly circular** — beware label leakage |
| PrimateAI | Partly (a benign *proxy*, not clinical labels) | **Medium** circularity |

➡️ **Forward reference:** Notebook **08** builds the de-circularization / benchmarking analysis
properly — using **orthogonal** truth sets like CFTR2 functional data and population frequency, so
supervised tools are not simply graded against their own homework.


## 4 · REVEL isn't one cut-point — it's a *graded* scale

A very common shortcut is: *"REVEL ≥ 0.75 → likely pathogenic, otherwise not."* That single cut
throws away information.

The ClinGen **Sequence Variant Interpretation** working group (**Pejaver et al. 2022**,
*AJHG*, **PMID 36413997**) *calibrated* REVEL against the **ACMG/AMP** evidence framework. They
showed a REVEL score maps to **graded tiers of pathogenic evidence strength** (Supporting →
Moderate → Strong), not a single yes/no line:

| REVEL score ≥ | ACMG pathogenic evidence strength (approx.) |
|---|---|
| **0.932** | **Strong** (PP3_Strong) |
| **0.773** | **Moderate** (PP3_Moderate) |
| **0.644** | **Supporting** (PP3_Supporting) |
| **0.290** | *below this* → **Benign** supporting evidence (BP4) |

(These break-points are approximate and are the ones used in the toolkit's teaching table.)

**Why it matters:** two variants at REVEL 0.65 and 0.95 are *both* "≥ 0.75-ish territory" under a
single cut, but the calibration says one is only **Supporting** evidence and the other is
**Strong**. Collapsing them to one label loses exactly the nuance a curator needs.

The toolkit deliberately ships a **single** binary cut (`tk.THRESHOLDS['revel']`) to keep the
`call_from_score` helper simple — but you should know the richer, graded reality exists.


In [3]:
# The toolkit's SIMPLE binary cut-points (one 'pathogenic' line, one 'benign' line):
tk.THRESHOLDS['revel']

{'path': 0.75, 'benign': 0.29}

## 6 · How to get the REAL REVEL scores

Download the **genome-wide REVEL table** from `https://sites.google.com/site/revelgenomics/`. It is keyed by **genomic coordinate** (`chr, pos, ref, alt`, GRCh37/38) — join on the coordinate, **not** the protein change (one protein change can arise from several codon changes).

> ⚠️ **CFTR minus-strand gotcha:** make sure ref/alt are on the same genomic strand as the downloaded table, or you silently match nothing (the CADD notebook shows the strand-flip trick).

## 8 · A taste of the graded idea — binning REVEL into the 4 Pejaver tiers

Instead of one line at 0.75, let's sort the DEMO variants into the **four calibrated tiers** from
section 4 using `pandas.cut`. This is a miniature version of what ACMG-style graded evidence looks
like in practice.


In [4]:
tier_edges  = [-np.inf, 0.290, 0.644, 0.773, 0.932, np.inf]
tier_labels = ['Benign-supporting (<0.290)', 'Indeterminate (0.290-0.644)',
               'Path Supporting (0.644-0.773)', 'Path Moderate (0.773-0.932)',
               'Path Strong (>=0.932)']
revel['revel_tier'] = pd.cut(revel['revel_score'], bins=tier_edges, labels=tier_labels, right=False)
tier_counts = revel['revel_tier'].value_counts().reindex(tier_labels).fillna(0).astype(int)
print('DEMO REVEL variants per Pejaver 2022 evidence tier:\n')
print(tier_counts.to_string())

DEMO REVEL variants per Pejaver 2022 evidence tier:

revel_tier
Benign-supporting (<0.290)       2
Indeterminate (0.290-0.644)      1
Path Supporting (0.644-0.773)    4
Path Moderate (0.773-0.932)      5
Path Strong (>=0.932)            1


Even in this tiny DEMO set you can see variants spread across **several** evidence strengths — the
detail a single 0.75 cut-point would flatten into just "pathogenic vs not".


## 7 · What the toolkit records about REVEL

`toolkit.py` keeps a registry entry per predictor — its learning type and **circularity** rating — which notebook 13 uses to decide which tools can be fairly benchmarked.

In [5]:
info = tk.TOOL_REGISTRY['REVEL']
for key, val in info.items():
    print(f'  {key:12s}: {val}')

  kind        : missense
  learning    : supervised
  signal      : random-forest ENSEMBLE of 13 scores, trained on curated pathogenic/benign labels
  circularity : HIGH (label lineage overlaps ClinVar/HGMD)
  pmid        : 27666373


## Key takeaways

1. **REVEL** is a **supervised** ensemble of 13 tools; score 0-1, higher = worse.
2. Its labels **share lineage with ClinVar/HGMD**, so grading it against ClinVar is **partly circular** (label leakage).
3. Read REVEL as a **graded** scale (Pejaver 2022), not a single 0.75 cut.
4. **DEMO** data — join the real genome-wide table on genomic coordinate.

**Next:** notebook 06 — **PrimateAI**, the semi-supervised alternative.